# 📱 Deteksi Kecanduan Smartphone
## Notebook 1: Pre-processing & Data Cleaning
---
Dataset: User Behavior Dataset  
Target: `User Behavior Class` (1–5, tingkat kecanduan smartphone)

### 1.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ Library berhasil diimport')

### 1.2 Load Dataset

In [ ]:
df = pd.read_csv('user_behavior_dataset.csv')

print(f'Shape dataset: {df.shape}')
print(f'Jumlah baris  : {df.shape[0]}')
print(f'Jumlah kolom  : {df.shape[1]}')
df.head(10)

### 1.3 Informasi Dataset

In [ ]:
print('=== Info Dataset ===')
df.info()

In [ ]:
print('=== Statistik Deskriptif ===')
df.describe().round(2)

### 1.4 Cek Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Persentase (%)': missing_pct.round(2)
})
print('=== Missing Values per Kolom ===')
print(missing_df)

if missing.sum() == 0:
    print('\n✅ Tidak ada missing values!')
else:
    print(f'\n⚠️  Total missing values: {missing.sum()}')

### 1.5 Cek Duplikasi Data

In [ ]:
duplikat = df.duplicated().sum()
print(f'Jumlah baris duplikat: {duplikat}')

if duplikat > 0:
    df = df.drop_duplicates()
    print(f'✅ {duplikat} baris duplikat dihapus. Shape baru: {df.shape}')
else:
    print('✅ Tidak ada data duplikat!')

### 1.6 Analisis Distribusi Target (User Behavior Class)

In [ ]:
class_labels = {
    1: 'Sangat Rendah',
    2: 'Rendah',
    3: 'Sedang',
    4: 'Tinggi',
    5: 'Sangat Tinggi'
}

class_dist = df['User Behavior Class'].value_counts().sort_index()
class_pct  = (class_dist / len(df) * 100).round(2)

print('=== Distribusi Kelas Kecanduan ===')
for k, v in class_dist.items():
    print(f'  Kelas {k} ({class_labels[k]:>14}): {v:>4} data ({class_pct[k]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71','#3498db','#f39c12','#e74c3c','#8e44ad']
axes[0].bar([class_labels[i] for i in class_dist.index], class_dist.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribusi Kelas Kecanduan Smartphone', fontweight='bold')
axes[0].set_xlabel('Tingkat Kecanduan')
axes[0].set_ylabel('Jumlah Data')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(class_dist.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

axes[1].pie(class_dist.values, labels=[class_labels[i] for i in class_dist.index],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Proporsi Kelas Kecanduan', fontweight='bold')

plt.tight_layout()
plt.savefig('01_distribusi_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gambar distribusi kelas disimpan.')

### 1.7 Deteksi Outlier (IQR Method)

In [ ]:
num_cols = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
            'Battery Drain (mAh/day)', 'Number of Apps Installed',
            'Data Usage (MB/day)', 'Age']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

outlier_summary = {}
for i, col in enumerate(num_cols):
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = n_out

    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}\n(Outlier: {n_out})', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Nilai')

plt.suptitle('Deteksi Outlier – Boxplot per Fitur Numerik', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('01_outlier_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Ringkasan Outlier (IQR Method) ===')
for col, n in outlier_summary.items():
    print(f'  {col:<40}: {n} outlier')

### 1.8 Hapus Kolom Tidak Relevan

In [ ]:
# Hapus User ID (bukan fitur prediktif)
df_clean = df.drop(columns=['User ID'])
print(f'Kolom setelah hapus User ID: {list(df_clean.columns)}')
print(f'Shape: {df_clean.shape}')

### 1.9 Encoding Variabel Kategorikal

In [ ]:
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Kolom kategorikal: {cat_cols}')

for col in cat_cols:
    print(f'\n  {col}: {df_clean[col].unique()}')

# Label Encoding untuk Gender
df_clean['Gender'] = df_clean['Gender'].map({'Male': 0, 'Female': 1})

# Label Encoding untuk Operating System
df_clean['Operating System'] = df_clean['Operating System'].map({'Android': 0, 'iOS': 1})

# One-Hot Encoding untuk Device Model
df_clean = pd.get_dummies(df_clean, columns=['Device Model'], drop_first=True, dtype=int)

print('\n✅ Encoding selesai')
print(f'Shape setelah encoding: {df_clean.shape}')
df_clean.head(3)

### 1.10 Simpan Data Hasil Cleaning

In [ ]:
df_clean.to_csv('data_cleaned.csv', index=False)
print('✅ Data bersih disimpan ke: data_cleaned.csv')
print(f'Jumlah fitur akhir : {df_clean.shape[1] - 1}')
print(f'Jumlah sampel      : {df_clean.shape[0]}')
print(f'Kolom: {list(df_clean.columns)}')